<a href="https://colab.research.google.com/github/gabscorreia/gcp-data-pipeline-serverless/blob/main/camada_tratamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**BIBLIOTECAS UTILIZADAS**

In [7]:
import pandas_gbq
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

**LEITURA DOS DADOS**

In [8]:
# esse trecho serve para listar os datasets, onde dá pra ver exatamente os nomes que o GCP registou.
client = bigquery.Client(project='strong-planet-484414-a0')
datasets = list(client.list_datasets())

for dataset in datasets:
  print(dataset.dataset_id)

ecommerce_bronze


In [9]:
# Configurações do projeto
project_id = "strong-planet-484414-a0"
dataset_id = "ecommerce_bronze"

# Inicializando o cliente do BigQuery
client = bigquery.Client(project=project_id)

In [10]:
# Lista todas as tabelas do dataset
tables = client.list_tables(dataset_id)
tabelas_dict = {}

In [11]:
for table in tables:
  tabela_nome = table.table_id
  print(f"A carregar a tabela: {tabela_nome}...")

  # Cria a query dinâmica para cada tabela
  query = f"SELECT * FROM `{project_id}.{dataset_id}.{tabela_nome}`"

  # Lê a tabela para um DataFrame do Pandas
  df = pandas_gbq.read_gbq(
      query, project_id=project_id, location="southamerica-east1"
  )
  tabelas_dict[tabela_nome] = df

print("\nTodas as tabelas foram carregadas com sucesso para o dicionário!")

A carregar a tabela: tb_category_name...
Downloading: 100%|██████████|
A carregar a tabela: tb_cliente...
Downloading: 100%|██████████|
A carregar a tabela: tb_geolocalizacao...
Downloading: 100%|██████████|
A carregar a tabela: tb_order_items...
Downloading: 100%|██████████|
A carregar a tabela: tb_order_payments...
Downloading: 100%|██████████|
A carregar a tabela: tb_orders_...
Downloading: 100%|██████████|
A carregar a tabela: tb_products_...
Downloading: 100%|██████████|
A carregar a tabela: tb_reviews...
Downloading: 100%|██████████|
A carregar a tabela: tb_sellers...
Downloading: 100%|██████████|

Todas as tabelas foram carregadas com sucesso para o dicionário!


**TRATAMENTO DOS DADOS**

In [12]:
tabelas_silver = {}

for tabela_nome, df in tabelas_dict.items():
  print(f"A processar e limpar a tabela: {tabela_nome}")

  # 1. Regra geral: Remove linhas totalmente duplicadas
  df_clean = df.drop_duplicates().copy()

  # 2. Regras específicas de tipagem e tratamento por tabela
  if tabela_nome == "tb_geolocalizacao":
    df_clean["geolocation_lat"] = pd.to_numeric(
        df_clean["geolocation_lat"], errors="coerce"
    )
    df_clean["geolocation_lng"] = pd.to_numeric(
        df_clean["geolocation_lng"], errors="coerce"
    )

  elif tabela_nome == "tb_orders_":
    timestamp_cols = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
    for col in timestamp_cols:
      if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

  elif tabela_nome == "tb_order_items":
    df_clean["price"] = pd.to_numeric(df_clean["price"], errors="coerce")
    df_clean["freight_value"] = pd.to_numeric(
        df_clean["freight_value"], errors="coerce"
    )

  elif tabela_nome == "tb_products_":
    if "product_category_name" in df_clean.columns:
      df_clean["product_category_name"] = df_clean[
          "product_category_name"
      ].fillna("desconhecido")

  # Guarda o DataFrame tratado no dicionário da camada Silver
  tabelas_silver[tabela_nome] = df_clean
  print(f" -> Concluído. Linhas finais: {len(df_clean)}")

print("\nTodas as tabelas foram limpas e preparadas com sucesso!")

A processar e limpar a tabela: tb_category_name
 -> Concluído. Linhas finais: 71
A processar e limpar a tabela: tb_cliente
 -> Concluído. Linhas finais: 99441
A processar e limpar a tabela: tb_geolocalizacao
 -> Concluído. Linhas finais: 738332
A processar e limpar a tabela: tb_order_items
 -> Concluído. Linhas finais: 112650
A processar e limpar a tabela: tb_order_payments
 -> Concluído. Linhas finais: 103886
A processar e limpar a tabela: tb_orders_
 -> Concluído. Linhas finais: 99441
A processar e limpar a tabela: tb_products_
 -> Concluído. Linhas finais: 32951
A processar e limpar a tabela: tb_reviews
 -> Concluído. Linhas finais: 104076
A processar e limpar a tabela: tb_sellers
 -> Concluído. Linhas finais: 3095

Todas as tabelas foram limpas e preparadas com sucesso!


In [13]:
# CARGA DOS DADOS TRATADOS PARA O DATASET SILVER NO BIGQUERY
dataset_silver = "ecommerce_silver"

for tabela_nome, df_clean in tabelas_silver.items():
  print(f"A carregar a tabela '{tabela_nome}' para a camada Silver...")

  df_clean.to_gbq(
      destination_table=f"{dataset_silver}.{tabela_nome}",
      project_id=project_id,
      if_exists="replace",  # Substitui a tabela caso já exista
      location="southamerica-east1",
  )

  print(f" -> Tabela '{tabela_nome}' salva com sucesso em {dataset_silver}!")

print(
    "\n=== PROCESSO DA CAMADA SILVER FINALIZADO: TODAS AS TABELAS FORAM"
    " GRAVADAS ==="
)

A carregar a tabela 'tb_category_name' para a camada Silver...


/tmp/ipykernel_644/1958452668.py:7: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_clean.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 8490.49it/s]
/tmp/ipykernel_644/1958452668.py:7: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_clean.to_gbq(


 -> Tabela 'tb_category_name' salva com sucesso em ecommerce_silver!
A carregar a tabela 'tb_cliente' para a camada Silver...


100%|██████████| 1/1 [00:00<00:00, 10058.28it/s]
/tmp/ipykernel_644/1958452668.py:7: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_clean.to_gbq(


 -> Tabela 'tb_cliente' salva com sucesso em ecommerce_silver!
A carregar a tabela 'tb_geolocalizacao' para a camada Silver...


...

 -> Tabela 'tb_products_' salva com sucesso em ecommerce_silver!
A carregar a tabela 'tb_reviews' para a camada Silver...


100%|██████████| 1/1 [00:00<00:00, 8490.49it/s]
/tmp/ipykernel_644/1958452668.py:7: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_clean.to_gbq(


 -> Tabela 'tb_reviews' salva com sucesso em ecommerce_silver!
A carregar a tabela 'tb_sellers' para a camada Silver...


100%|██████████| 1/1 [00:00<00:00, 11096.04it/s]

 -> Tabela 'tb_sellers' salva com sucesso em ecommerce_silver!

=== PROCESSO DA CAMADA SILVER FINALIZADO: TODAS AS TABELAS FORAM GRAVADAS ===
